In [11]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("Imports completed successfully!")

Imports completed successfully!


In [12]:
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Test: (19506, 45)


In [13]:
print("Columns:")
print(train_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_df.head())

print("\nData types:")
print(train_df.dtypes)

Columns:
['appID', 'name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'dlc_count', 'detailed_description', 'short_description', 'supported_languages', 'full_audio_languages', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'user_score', 'positive', 'negative', 'score_rank', 'achievements', 'recommendations', 'notes', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'screenshots', 'movies', 'packages', 'release_year', 'rank_in_year', 'games_in_year', 'hit']

First 5 rows:


,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,...,categories,genres,tags,screenshots,movies,packages,release_year,rank_in_year,games_in_year,hit
0,2888030,Coyote: An Old West Vignette,2024-05-30,0 - 0,0,0,0.00,0,Coyote: An Old West Vignette is a tiny point-a...,Coyote: An Old West Vignette is a tiny point-a...,...,"[Single-player, Full controller support, Steam...","[Adventure, Indie, Free To Play]",[],[https://shared.akamai.steamstatic.com/store_i...,[],[],2024,17032.0,20031,0
1,753420,Dungreed,2018-02-14,200000 - 500000,123,0,4.99,1,Dungreed is a 2D side-scrolling action game wi...,Dungreed is a 2D side-scrolling action game wi...,...,"[Single-player, Steam Achievements, Full contr...","[Action, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Dungreed"", ""description"": """", ...",2018,145.0,7461,1
2,747360,Gray Dawn,2018-06-07,50000 - 100000,1,0,1.99,0,You will be entangled in a terrifying adventur...,Embark on a terrifying adventure of a priest a...,...,"[Single-player, Steam Achievements, Full contr...","[Adventure, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Gray Dawn"", ""description"": """",...",2018,510.0,7461,1
3,2463530,Vacation Paradise: Florida Collector's Edition,2023-07-22,0 - 20000,1,0,6.49,0,Take a Vacation with us to the sunshine state ...,"Take a Vacation in Florida! Beautiful beaches,...",...,"[Single-player, Family Sharing]","[Action, Adventure, Casual, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Vacation Paradise: Florida Col...",2023,14369.0,14596,0
4,1269540,Slash Roll,2020-08-26,0 - 20000,0,0,2.99,0,Remember to wishlist the game About the Game A...,"After slaying some spire, the taverns of Slash...",...,"[Single-player, Steam Achievements, Steam Clou...","[Indie, Strategy]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Slash Roll"", ""description"": """"...",2020,5637.0,8804,0



Data types:
appID                                  str
name                                   str
release_date                datetime64[us]
estimated_owners                       str
peak_ccu                             int64
required_age                         int64
price                              float64
dlc_count                            int64
detailed_description                   str
short_description                      str
supported_languages                 object
full_audio_languages                object
reviews                                str
header_image                           str
website                                str
support_url                            str
support_email                          str
windows                               bool
mac                                   bool
linux                                 bool
metacritic_score                     int64
metacritic_url                         str
user_score                           int6

In [14]:
# Create count features from categories and genres
for df in [train_df, val_df, test_df]:
    df["categories_count"] = df["categories"].apply(len)
    df["genres_count"] = df["genres"].apply(len)

# Select the best features from M3
features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "categories_count",
    "genres_count"
]

target = "hit"

# Separate features and target
X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Features:", features)
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTraining target percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target distribution:")
print(y_val.value_counts())

Features: ['price', 'windows', 'mac', 'linux', 'required_age', 'categories_count', 'genres_count']

Training target distribution:
hit
0    43988
1    14528
Name: count, dtype: int64

Training target percentages:
hit
0    75.17
1    24.83
Name: proportion, dtype: float64

Validation target distribution:
hit
0    14598
1     4908
Name: count, dtype: int64


In [15]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Train a dummy classifier
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

# Predict on validation data
dummy_predictions = dummy_model.predict(X_val)

# Evaluate the model
print("Dummy Classifier")
print("-----------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, dummy_predictions))

print("\nPrecision:", precision_score(y_val, dummy_predictions, zero_division=0))
print("Recall:", recall_score(y_val, dummy_predictions, zero_division=0))
print("F1:", f1_score(y_val, dummy_predictions, zero_division=0))

Dummy Classifier
-----------------

Confusion Matrix:
[[14598     0]
 [ 4908     0]]

Precision: 0.0
Recall: 0.0
F1: 0.0


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Create logistic regression model
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Train the model
logistic_model.fit(X_train, y_train)

# Predict on validation data
logistic_predictions = logistic_model.predict(X_val)

# Evaluate the model
print("Logistic Regression")
print("-------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, logistic_predictions))

print("\nClassification Report:")
print(classification_report(y_val, logistic_predictions, zero_division=0))

print("\nHit class metrics:")
print("Precision:", precision_score(y_val, logistic_predictions))
print("Recall:", recall_score(y_val, logistic_predictions))
print("F1:", f1_score(y_val, logistic_predictions))

Logistic Regression
-------------------

Confusion Matrix:
[[14007   591]
 [ 4187   721]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.96      0.85     14598
           1       0.55      0.15      0.23      4908

    accuracy                           0.76     19506
   macro avg       0.66      0.55      0.54     19506
weighted avg       0.71      0.76      0.70     19506


Hit class metrics:
Precision: 0.5495426829268293
Recall: 0.14690301548492257
F1: 0.23183279742765273


### Pixel Harbor's assumptions

Pixel Harbor receives hundreds of game ideas each year but can only finance approximately 10% of them. Therefore, the model should recommend a limited number of games rather than accepting every game that has some chance of becoming a hit.

We assume that a false alarm, where Pixel Harbor finances a game that becomes a flop, has a cost of 1 point. A missed hit, where Pixel Harbor rejects a game that would have become successful, has a cost of 5 points. We consider a missed hit more expensive because Pixel Harbor loses the opportunity to invest in a potentially successful game.

These values are estimates used to compare different classification thresholds. The goal is to find a threshold that balances the cost of incorrect decisions with Pixel Harbor's limited financing capacity.

In [17]:
import pandas as pd
import numpy as np

# Get the probability that each validation game is a hit
val_probabilities = logistic_model.predict_proba(X_val)[:, 1]

# Pixel Harbor's estimated costs
false_alarm_cost = 1
missed_hit_cost = 5

# Test different thresholds
thresholds = np.arange(0.05, 1.00, 0.05)

results = []

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    precision = precision_score(y_val, predictions, zero_division=0)
    recall = recall_score(y_val, predictions, zero_division=0)
    f1 = f1_score(y_val, predictions, zero_division=0)

    total_cost = (
        fp * false_alarm_cost +
        fn * missed_hit_cost
    )

    results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Games recommended": int(predictions.sum()),
        "False alarms": fp,
        "Missed hits": fn,
        "Total cost": total_cost
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.round(3).to_string(index=False))

 Threshold  Precision  Recall    F1  Games recommended  False alarms  Missed hits  Total cost
      0.05      0.252   1.000 0.402              19498         14591            1       14596
      0.10      0.264   1.000 0.418              18562         13656            2       13666
      0.15      0.286   0.955 0.440              16395         11706          219       12801
      0.20      0.356   0.758 0.485              10453          6731         1186       12661
      0.25      0.406   0.559 0.470               6752          4009         2165       14834
      0.30      0.430   0.391 0.409               4460          2542         2990       17492
      0.35      0.450   0.291 0.353               3168          1741         3481       19146
      0.40      0.487   0.234 0.316               2360          1211         3759       20006
      0.45      0.526   0.186 0.275               1739           824         3993       20789
      0.50      0.550   0.147 0.232               1312      

In [18]:
# Pixel Harbor can finance at most 10% of games
financing_capacity = 0.10

max_games = int(len(y_val) * financing_capacity)

# Find the probability threshold for the top 10%
sorted_probabilities = np.sort(val_probabilities)[::-1]
capacity_threshold = sorted_probabilities[max_games - 1]

# Predict using the capacity threshold
capacity_predictions = (
    val_probabilities >= capacity_threshold
).astype(int)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(
    y_val, capacity_predictions, labels=[0, 1]
).ravel()

capacity_precision = precision_score(
    y_val, capacity_predictions, zero_division=0
)
capacity_recall = recall_score(
    y_val, capacity_predictions, zero_division=0
)
capacity_f1 = f1_score(
    y_val, capacity_predictions, zero_division=0
)

capacity_cost = fp * false_alarm_cost + fn * missed_hit_cost

print("Pixel Harbor's 10% Financing Capacity")
print("--------------------------------------")
print("Threshold:", round(capacity_threshold, 4))
print("Games recommended:", int(capacity_predictions.sum()))
print("Financing capacity:", max_games)
print("Precision:", round(capacity_precision, 3))
print("Recall:", round(capacity_recall, 3))
print("F1:", round(capacity_f1, 3))
print("False alarms:", fp)
print("Missed hits:", fn)
print("Total cost:", capacity_cost)

Pixel Harbor's 10% Financing Capacity
--------------------------------------
Threshold: 0.4362
Games recommended: 1950
Financing capacity: 1950
Precision: 0.51
Recall: 0.203
F1: 0.29
False alarms: 956
Missed hits: 3914
Total cost: 20526


### Choosing the probability threshold

The threshold of 0.20 produced the highest F1-score of 0.485 and the lowest estimated cost among the tested thresholds, with a total cost of 12,661 points. However, it recommended 10,453 games, which is approximately 53.6% of the validation dataset. This is far above Pixel Harbor's financing capacity of 10%.

We therefore selected a threshold of 0.4362, which recommends exactly 1,950 games, corresponding to 10% of the validation dataset. This threshold gives a precision of 0.510, recall of 0.203 and F1-score of 0.290.

Although the estimated cost increases to 20,526 points, this threshold respects Pixel Harbor's limited budget. We prioritize the financing capacity over the highest F1-score because the company cannot finance every game recommended by the model.

The selected threshold for the app's LOVANDE recommendation is **0.4362**. Games with a predicted hit probability equal to or above this value are classified as LOVANDE.

In [19]:
# Get hit probabilities for the test dataset
test_probabilities = logistic_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
final_threshold = capacity_threshold

test_predictions = (
    test_probabilities >= final_threshold
).astype(int)

# Confusion matrix
test_cm = confusion_matrix(
    y_test, test_predictions, labels=[0, 1]
)

# Calculate precision and recall
test_precision = precision_score(
    y_test, test_predictions, zero_division=0
)

test_recall = recall_score(
    y_test, test_predictions, zero_division=0
)

print("Final Test Evaluation")
print("---------------------")
print("Selected threshold:", round(final_threshold, 4))

print("\nConfusion Matrix:")
print(test_cm)

print("\nPrecision:", round(test_precision, 3))
print("Recall:", round(test_recall, 3))

print("\nGames recommended:", int(test_predictions.sum()))

Final Test Evaluation
---------------------
Selected threshold: 0.4362

Confusion Matrix:
[[13615   937]
 [ 3912  1042]]

Precision: 0.527
Recall: 0.21

Games recommended: 1979


### Final evaluation and conclusion

The final model was evaluated on the test dataset using the selected probability threshold of 0.4362. The confusion matrix shows 1,042 true positives, 13,615 true negatives, 937 false alarms and 3,912 missed hits.

The model achieved a precision of 0.527 and a recall of 0.210. This means that approximately 52.7% of the games recommended as LOVANDE were actual hits. The model identified 21% of all actual hits in the test dataset.

The model recommended 1,979 games, which is approximately 10.1% of the test dataset. This is close to Pixel Harbor's financing capacity of 10%. The small difference is expected because the threshold was selected using the validation dataset and was not adjusted using the test results.

Overall, the model can help Pixel Harbor identify a smaller group of potentially successful games. However, its low recall means that it misses many games that eventually become hits. The predictions should therefore be used as decision support rather than as the only factor when choosing which games to finance.

For the LOVANDE app, the selected threshold is **0.4362**. A game is labelled **LOVANDE** when its predicted hit probability is at least 0.4362. Otherwise, it is labelled **REKOMMENDERAS INTE**.